# Crecimiento poblacional logístico

Autor: Guillermo Hernández Ruiz de Esparza

Asesor: Dr. Edgar Ali Ramos Gómez

Proyecto: Implementación de herramientas computacionales mediante Python para la enseñanza de ecuaciones diferenciales

Facultad de Ingenieria, UNAM

## Objetivos

* Conocer los métodos de Euler y Runge-Kutta de 4 orden como alternativa para resolver ecuaciones diferenciales de primer orden
* Comprender la relacion entre los métodos numéricos y la solución analítica de una ecuación diferencial
* Identificar como afecta la variación del tamaño de paso $h$ en la aproximación de la solución analítica
* Implementar una solución analítica y numerica de la ecuación diferencial y graficarla, en Python

## Teoría

Modelar el crecimiento poblacional es un problema que tiene aplicaciones en biología, ecología, economía, y otras áreas. Los modelos de crecimiento poblacional permiten predecir cómo cambiará la población de una especie a lo largo del tiempo, considerando factores como la tasa de natalidad, la tasa de mortalidad, capacidad de carga, migración, emigración, entre otros.

Existen varios modelos de crecimiento poblacional:
* Crecimiento exponencial: aumento de la población sin restricciones
* Crecimiento logístico: aumento de la población con un límite de capacidad de carga (es el que se estudiará en este notebook)
* Crecimiento con migración: aumento de la población considerando la entrada de individuos de otras poblaciones
* Crecimiento con emigración: aumento de la población considerando la salida de individuos hacia otras poblaciones

---
Cualquier modelo de crecimiento poblacional tiene una tasa de crecimiento poblacional, esta tasa es el cambio de la población en un periodo de tiempo. en las ecuaciones se representa como $r$ y se calcula como $r=\frac{dP/dt}{P}=\frac{P_{t+1}-P_t}{P_t}$.

Para mostrar como se puede obtener esta tasa vamos a usar el siguiente ejemplo:
* Un tipo de celula realiza mitosis cada 10 minutos
* La celula tiene una vida de 11 minutos
* Cuando se empieza a estudiar la población de celulas, solo se tiene una celula

Para poder seguir las celulas se usara la nomenclatura X_G, donde G es la generacion de la celula, y X es el numero de celula. Las hijas de una celula estan con lineas punteadas, cuando una celula continua a la siguiente generacion se muestra con una linea solida.

```mermaid
flowchart TD

    %% =========================
    %% t = 0
    %% =========================
    subgraph T0["t = 0"]
        A0["1_1"]
    end

    %% =========================
    %% t = 10
    %% =========================
    subgraph T10["t = 10"]
        A10["1_1"]
        B10["1_2"]
        C10["2_2"]
    end

    %% =========================
    %% t = 20
    %% =========================
    subgraph T20["t = 20"]
        B20["1_2"]
        C20["2_2"]
        D20["1_3"]
        E20["2_3"]
        F20["3_3"]
        G20["4_3"]
    end

    %% =========================
    %% t = 30
    %% =========================
    subgraph T30["t = 30"]
        D30["1_3"]
        E30["2_3"]
        F30["3_3"]
        G30["4_3"]

        H30["1_4"]
        I30["2_4"]
        J30["3_4"]
        K30["4_4"]
        L30["5_4"]
        M30["6_4"]
        N30["7_4"]
        O30["8_4"]
    end

    %% =====================================
    %% CONTINUIDAD - LÍNEAS SÓLIDAS
    %% =====================================

    A0 --> A10

    B10 --> B20
    C10 --> C20

    D20 --> D30
    E20 --> E30
    F20 --> F30
    G20 --> G30

    %% =====================================
    %% NACIMIENTO DE HIJAS - LÍNEAS PUNTEADAS
    %% =====================================

    A0 -.-> B10
    A0 -.-> C10

    B10 -.-> D20
    B10 -.-> E20

    C10 -.-> F20
    C10 -.-> G20

    D20 -.-> H30
    D20 -.-> I30

    E20 -.-> J30
    E20 -.-> K30

    F20 -.-> L30
    F20 -.-> M30

    G20 -.-> N30
    G20 -.-> O30
```

En este caso para calcular la tasa de crecimiento poblacional se usa:
* $P_{10}=3$
* $P_{0}=1$
$$r=\frac{P_{10}-P_0}{P_0}=\frac{3-1}{1}=2$$
Pueden existir casos donde la tasa de crecimiento sea negativa, o sea 0; pero el modelo de crecimiento poblacional logístico tradicional solo considera tasas de crecimiento positivas.

Si se utiliza solo la tasa de crecimiento se tiene el modelo exponencial
$$\frac{dP}{dt}=rP$$
Pero ese modelo no es realista en la mayoria de casos ya que no toma en cuenta las limitacines de espacio o recursos del medio ambiente. Para modelar estas limitaciones se tiene la capacidad de carga del ambiente $K$, que es el número máximo de individuos que un ambiente puede sostener. Con la capacidad de carga se agrega un factor de amortiguamiento que reduce la tasa de crecimiento poblacional a medida que la población se acerca a la capacidad de carga y cuando la población es igual a la capacidad de carga, la población deja de crecer. 

Al introducir la capacidad de carga en el modelo exponencial se obtiene el modelo logístico que tiende siempre a la capacidad de carga.
$$\frac{dP}{dt}=rP\left(1-\frac{P}{K}\right)$$
La capacidad de carga solo puede ser un valor positivo ya que representa el número máximo de individuos que un ambiente puede sostener. En algunos libros el modelo se modifica y se representa como:
$$\frac{dP}{dt}=P\left(a-bP\right)$$
Para este cuaderno se va a utilizar la primera forma del modelo logístico, ya que representa explicitamente $r$ y $K$.

Ya que se tiene la ecuación diferencial se puede clasificar, el primer paso para analizarla es extender la ecuación:
$$ \frac{dP}{dt}=rP-\frac{r}{K}P^2$$
* La variable $P$ esta elevada a la segunda potencia, por lo que no es una ED lineal
* Las derivadas solo dependen de 1 variable independiente, por lo que es una ED ordinaria
* El orden de la derivada de mayor orden es 1, por lo que es una ED de primer orden

El modelo logístico es una ecuación diferencial ordinaria de primer orden no lineal.

---
El último paso antes de empezar a resolver la ecuacion es analizar la ecuacion para entender el comportamiento esperado. Se tienen que identificar los coeficientes y variables y que rangos puede tomar.
* $r$: tasa de crecimiento poblacional, solo puede ser positiva
* $K$: capacidad de carga, solo puede ser positiva
* $P$: población, solo puede ser positiva
* $P(0)$: población inicial, solo puede ser positiva; puede ser mayor, menor o igual a $K$
* $t$: tiempo, solo puede ser positivo

Para entender el comportamiento del modelo vamos a analizar que pasa cuando $t \to \infty$.

Mientras $r$ sea más grande, la población cambiará más rápido; mientras $r$ sea más pequeño, la población cambiará más lentamente. 

Mientras $K$ sea más lejano a $P$ se tardará más tiempo en llegar a la capacidad de carga; mientras $K$ es más cercano a $P$ se tardará menos tiempo en llegar a la capacidad de carga. Cuando $P=K$ el factor de amortiguamiento $\left(1-\frac{P}{K}\right)$ se hace 0 y la población deja de crecer sin importar el resto de parámetros.

El último parametro es $P(0)$, esta tiene 3 posibles casos:
* $P(0)<K$: la población inicial es menor a la capacidad de carga, por lo que la población crecerá hasta llegar a $K$
* $P(0)=K$: la población inicial es igual a la capacidad de carga (población en equilibrio), la solución particular es una constante y la población no crecerá ni decrecerá
* $P(0)>K$: la población inicial es mayor a la capacidad de carga (sobrepoblación), por lo que la población decrecerá hasta llegar a $K$

---
(Zill et al., 2009)

El modelo logístico es el siguiente:
$$\frac{dP}{dt}=rP\left(1-\frac{P}{K}\right)$$

Se puede resolver por separacion de variables y el resultado es:
$$P(t)=\frac{K}{C_1e^{-rt}+1}$$

Al aplicar la condición inicial $P(0)=P_0$ se obtiene:
$$P_0=\frac{K}{C_1e^{0}+1}$$

Y al despejar $C_1$ se obtiene:
$$C_1=\frac{K}{P_0}-1$$

Con esto la solución particular es:
$$P(t)=\frac{K}{\left(\frac{K}{P_0}-1\right)e^{-rt}+1}$$

## Implementación en Python

### Bibliotecas

En esta sección se importan las bibliotecas necesarias para trabajar con ecuaciones diferenciales, graficar y crear la interfaz interactiva.

In [ ]:
import sympy as sp # Cálculo simbólico
import numpy as np # Cálculo numérico optimizado
import pandas as pd # Crear tablas de datos

import matplotlib.pyplot as plt # Creación de gráficas
import matplotlib.colors as mcolors # Colores para el campo direccional

import ipywidgets as widgets # Hacer la interfaz interactiva
from IPython.display import display, Math, HTML # Hacer la interfaz interactiva

# Configuración para optimizar las gráficas en notebooks
%matplotlib inline

# Configuración para mostrar tablas completas en notebooks
pd.set_option('display.max_rows', None)

In [ ]:
# Ejecuta esta celda solo si al ejecutar la celda anterior se muestra un error despues reinicia el cuaderno y vuelve a ejecutar la celda anterior
!python.exe -m pip install --upgrade pip -q
%pip install sympy -q
%pip install numpy -q
%pip install pandas -q
%pip install matplotlib -q
%pip install ipywidgets -q
%pip install ipython -q

### Definición de variables simbólicas y la ecuación diferencial

En esta sección se definen las variables simbólicas con SymPy y se define la ecuación diferencial que describe el modelo de crecimiento poblacional con límite de capacidad de carga.

En la teoría ya definimos la ecuación diferencial que describe el crecimiento poblacional con límite de capacidad de carga.
$$\frac{dP}{dt} = f(t, P) = rP\left(1-\frac{P}{K}\right)$$
 
Con esto podemos identificar las variables simbólicas que se necesitan:

|Variable|Descripción|Tipo en Sympy|
|---|---|---|
|$t$|Tiempo|Símbolo|
|$r$|Tasa de crecimiento poblacional|Símbolo|
|$K$|Capacidad de carga|Símbolo|
|$P$|Tamaño de la población en $t$|Función de $t$|

In [ ]:
t = sp.symbols('t')
r = sp.symbols('r')
K = sp.symbols('K')
P = sp.Function('P')(t)

In [ ]:
f = r*P*(1-P/K)

### Funciones para aplicar los metodos numéricos (Euler y RK4)

En esta seccion se implementan los métodos numéricos de Euler y Runge-Kutta de 4 orden

In [ ]:
def euler(t_0, t_max, P_0, h, f):
    # Definir las listas para almacenar los valores de t y P
    t = [t_0]
    P_numerica = [P_0]
    i = 0

    while t[-1] < t_max:
        t_i = t[i] # Obtener t_i
        t_siguiente = np.round(t_i + h, 2) # Calcular t_{i+1} y redondear
        t.append(t_siguiente) # Agregar t_{i+1} a la lista de t

        P_i = P_numerica[i] # Obtener P_i
        P_siguiente = P_i + h*f(t_i, P_i) # Aplicar la funcion de recurrencia del método de Euler
        P_siguiente = np.round(P_siguiente, 4) # Redondear P_{i+1}
        P_numerica.append(P_siguiente) # Agregar P_{i+1} a la lista de P_numerica
        
        i += 1 # Incrementar el índice i para la siguiente iteración

    # Eliminar infinitos de la lista de P_numerica
    P_numerica = [np.nan if np.isinf(x) else x for x in P_numerica]

    return pd.DataFrame({'t': t, 'P_numerica': P_numerica}) # Devolver un DataFrame con los resultados

In [ ]:
def RK4(t_0, t_max, P_0, h, f):
    # Definir las listas para almacenar los valores de t, k1, k2, k3, k4 y P
    t = [t_0]
    k1 = []
    k2 = []
    k3 = []
    k4 = []
    P_numerica = [P_0]
    i = 0

    while t[-1] < t_max:
        t_i = t[i] # Obtener t_i
        t_siguiente = np.round(t_i + h, 2) # Calcular t_{i+1} y redondear
        t.append(t_siguiente) # Agregar t_{i+1} a la lista de t
        P_i = P_numerica[i] # Obtener P_i

        k1_i = f(t_i, P_i) # Aplicar la funcion de recurrencia para k1 de RK4
        k1_i = np.round(k1_i, 4) # Redondear k1_i
        k1.append(k1_i) # Agregar k1_i a la lista de k1

        k2_i = f(t_i + h/2, P_i + h*k1_i/2) # Aplicar la funcion de recurrencia para k2 de RK4
        k2_i = np.round(k2_i, 4) # Redondear k2_i
        k2.append(k2_i) # Agregar k2_i a la lista de k2

        k3_i = f(t_i + h/2, P_i + h*k2_i/2) # Aplicar la funcion de recurrencia para k3 de RK4
        k3_i = np.round(k3_i, 4) # Redondear k3_i
        k3.append(k3_i) # Agregar k3_i a la lista de k3

        k4_i = f(t_i + h, P_i + h*k3_i) # Aplicar la funcion de recurrencia para k4 de RK4
        k4_i = np.round(k4_i, 4) # Redondear k4_i
        k4.append(k4_i) # Agregar k4_i a la lista de k4

        P_siguiente = P_i + h*(k1_i + 2*k2_i + 2*k3_i + k4_i)/6 # Aplicar la funcion de recurrencia del método de RK4
        P_siguiente = np.round(P_siguiente, 4) # Redondear P_{i+1}
        P_numerica.append(P_siguiente) # Agregar P_{i+1} a la lista de P_numerica

        i += 1 # Incrementar el índice i para la siguiente iteración

    # Redondear las listas de k1, k2, k3 y k4 para que tengan la misma longitud que t y P_numerica
    k1.append(np.nan)
    k2.append(np.nan)
    k3.append(np.nan)
    k4.append(np.nan)

    # eliminar infinitos de las listas de k1, k2, k3, k4 y P_numerica
    k1 = [np.nan if np.isinf(x) else x for x in k1]
    k2 = [np.nan if np.isinf(x) else x for x in k2]
    k3 = [np.nan if np.isinf(x) else x for x in k3]
    k4 = [np.nan if np.isinf(x) else x for x in k4]
    P_numerica = [np.nan if np.isinf(x) else x for x in P_numerica]

    return pd.DataFrame({'t': t, 'k1': k1, 'k2': k2, 'k3': k3, 'k4': k4, 'P_numerica': P_numerica}) # Devolver un DataFrame con los resultados

### Funciones para calcular la solución analítica y calcular los errores

En esta sección se implementa la función para calcular la solución analítica y calcular los errores de los métodos numéricos con respecto a la solución analítica.

In [ ]:
def f_analítica(t, P0, K, r):
    return K/((K/P0 - 1)*np.exp(-r*t) + 1)

In [ ]:
def solucion_analitica(t_0, t_max, h, P0, K, r):
    # Definir las listas para almacenar los valores de t y P
    t = [t_0]
    P_analitica = []

    while t[-1] < t_max + h:
        t_i = t[-1] # Obtener t_i
        P_i = f_analítica(t_i, P0, K, r) # Calcular P_i usando la función analítica
        P_i = np.round(P_i, 4) # Redondear P_i
        P_analitica.append(P_i) # Agregar P_i a la lista de P_analitica

        t_siguiente = np.round(t_i + h, 2) # Calcular t_{i+1} y redondear
        t.append(t_siguiente) # Agregar t_{i+1} a la lista de t

    return pd.DataFrame({'t': t[:-1], 'P_analitica': P_analitica}) # Devolver un DataFrame con los resultados

In [ ]:
def error_absoluto(P_numerica, P_analitica):
    datos = pd.merge(P_numerica, P_analitica, on='t', how='inner')  # Unir los DataFrames por la columna 't'
    datos['error_absoluto'] = np.abs(datos['P_numerica'] - datos['P_analitica'])  # Calcular el error absoluto
    datos['error_absoluto'] = datos['error_absoluto'].replace([np.inf, -np.inf], np.nan)  # Reemplazar inf y -inf con NaN
    datos['error_absoluto'] = datos['error_absoluto'].round(4)  # Redondear el error absoluto a 4 decimales
    return datos

In [ ]:
def error_porcentual(P_numerica, P_analitica):
    datos = pd.merge(P_numerica, P_analitica, on='t', how='inner')  # Unir los DataFrames por la columna 't'
    datos['error_porcentual'] = np.abs((datos['P_numerica'] - datos['P_analitica']) / datos['P_analitica']) * 100  # Calcular el error porcentual
    datos['error_porcentual'] = datos['error_porcentual'].replace([np.inf, -np.inf], np.nan)  # Reemplazar inf y -inf con NaN
    datos['error_porcentual'] = datos['error_porcentual'].round(4)  # Redondear el error porcentual a 4 decimales
    return datos

### Función para graficar la solución analítica, la solución numérica y los errores

En esta sección se implementa la función para graficar los resultados

In [ ]:
def graficar(datos_euler, datos_RK4, sol_analitica, t_0, t_max, h, P0, K, r):
    fig = plt.figure(figsize=(15, 9))
    gs = fig.add_gridspec(3, 2)

    ax1 = fig.add_subplot(gs[0, :])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[1, 1])
    ax4 = fig.add_subplot(gs[2, 0])
    ax5 = fig.add_subplot(gs[2, 1])

    ax1.plot(np.linspace(t_0, t_max, 100), [f_analítica(t, P0, K, r) for t in np.linspace(t_0, t_max, 100)], color="blue", alpha=0.5)
    ax1.plot(sol_analitica['t'], sol_analitica['P_analitica'], label="Solución analítica", color='blue', marker='o', markersize=4, linestyle='None')
    ax1.plot(datos_euler['t'], datos_euler['P_numerica'], label=f"Euler (h = {h})", color="red", marker='v', markersize=4, linestyle='None')
    ax1.plot(datos_RK4['t'], datos_RK4['P_numerica'], label=f"RK4 (h = {h})", color="green", marker='v', markersize=4, linestyle='None')
    ax1.set_title("Solución analítica vs. métodos numéricos")
    ax1.set_xlabel("t")
    ax1.set_ylabel("P(t)")
    ax1.set_xlim(t_0, t_max)
    ax1.grid()
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.legend()

    ax2.plot(datos_euler['t'], datos_euler['error_absoluto'], color="black", marker="o", markersize=4, linestyle='None')
    ax2.set_title("Error absoluto Euler")
    ax2.set_xlabel("t")
    ax2.set_ylabel("E_a")
    ax2.set_xlim(t_0, t_max)
    ax2.grid()
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    ax3.plot(datos_euler['t'], datos_euler['error_porcentual'], color="black", marker="o", markersize=4, linestyle='None')
    ax3.set_title("Error porcentual Euler")
    ax3.set_xlabel("t")
    ax3.set_ylabel("E_p")
    ax3.set_ylim(0, 100)
    ax3.set_xlim(t_0, t_max)
    ax3.set_yticks(np.linspace(0, 100, 11))
    ax3.grid()
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)

    ax4.plot(datos_RK4['t'], datos_RK4['error_absoluto'], color="black", marker="o", markersize=4, linestyle='None')
    ax4.set_title("Error absoluto RK4")
    ax4.set_xlabel("t")
    ax4.set_ylabel("E_a")
    ax4.set_xlim(t_0, t_max)
    ax4.grid()
    ax4.spines['top'].set_visible(False)
    ax4.spines['right'].set_visible(False)

    ax5.plot(datos_RK4['t'], datos_RK4['error_porcentual'], color="black", marker="o", markersize=4, linestyle='None')
    ax5.set_title("Error porcentual RK4")
    ax5.set_xlabel("t")
    ax5.set_ylabel("E_p")
    ax5.set_ylim(0, 100)
    ax5.set_xlim(t_0, t_max)
    ax5.set_yticks(np.linspace(0, 100, 11))
    ax5.grid()
    ax5.spines['top'].set_visible(False)
    ax5.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.close(fig)

    return fig

## Laboratorio

### Creación de la interfaz

En esta sección se desarrolla la interfaz interactiva junto con todos los componentes necesarios para mostrar los resultados. Se incorporan sliders para modificar los parámetros de la ecuación diferencial y elementos para visualizar los resultados.

In [ ]:
# En esta celda se crean los sliders para los parámetros de la ecuación diferencial
slider_r = widgets.FloatSlider(
    value=0.1,
    min=0,
    max=5.0,
    step=0.1,
    description='r:',
    continuous_update=False
)

slider_K = widgets.IntSlider(
    value=100,
    min=0,
    max=2000,
    step=10,
    description='K:',
    continuous_update=False
)

slider_P0 = widgets.IntSlider(
    value=10,
    min=0,
    max=500,
    step=10,
    description='P(0):',
    continuous_update=False
)

slider_t_max = widgets.IntSlider(
    value=60,
    min=5,
    max=120,
    step=5,
    description='Rango de tiempo graficado:',
    continuous_update=False,
    style={'description_width': '200px'},
    layout=widgets.Layout(width='450px')
)

slider_h = widgets.FloatSlider(
    value=1,
    min=0.1,
    max=5,
    step=0.1,
    description='h:',
    continuous_update=False
)

checkbox_mostrar_tablas = widgets.Checkbox(
    value=False,
    description='Mostrar tablas de resultados',
    disabled=False
)

In [ ]:
# En esta celda se crean los componentes para mostrar los resultados
salida_modelo_sin_parametros = widgets.Output()
salida_fr_euler = widgets.Output()
salida_fr_RK4 = widgets.Output()
salida_grafica = widgets.Output()
salida_euler = widgets.Output()
salida_RK4 = widgets.Output()

In [ ]:
# En esta celda se configural los componentes que no cambian
with salida_modelo_sin_parametros:
    display(Math(sp.latex(sp.Eq(P.diff(t), f))))

In [ ]:
# En esta celda se define la estructura de la interfaz interactiva, organizando los sliders y las salidas en filas y columnas
controles_parametros = widgets.HBox([
    slider_r,
    slider_K,
    slider_P0
])

controles_grafica = widgets.HBox([
    slider_t_max
])

controles_metodos = widgets.HBox([
    slider_h,
    checkbox_mostrar_tablas
])

controles = widgets.VBox([
    controles_parametros,
    controles_grafica,
    controles_metodos
])

panel_analisis = widgets.VBox([
    widgets.HTML("<h3>Ecuación del modelo sin parámetros</h3>"),
    salida_modelo_sin_parametros,
    widgets.HTML("<h3>Función de recurrencia Euler</h3>"),
    salida_fr_euler,
    widgets.HTML("<h3>Función de recurrencia RK4</h3>"),
    salida_fr_RK4
])

panel_grafica = widgets.VBox([
    widgets.HTML("<h3>Gráfica de la solución</h3>"),
    salida_grafica
])

resultados = widgets.GridBox(
    children=[
        panel_analisis,
        panel_grafica
    ],
    layout=widgets.Layout(
        width='100%',
        grid_template_columns='25% 74%',
        grid_template_rows='auto',
        grid_gap='1%'
    )
)

panel_euler = widgets.VBox([
    widgets.HTML("<h3>Tabla de resultados Euler</h3>"),
    salida_euler
])

panel_RK4 = widgets.VBox([
    widgets.HTML("<h3>Tabla de resultados RK4</h3>"),
    salida_RK4
])

tablas = widgets.GridBox(
    children=[
        panel_euler,
        panel_RK4
    ],
    layout=widgets.Layout(
        width='100%',
        grid_template_columns='40% 55%',
        grid_template_rows='auto',
        grid_gap='5%'
    )
)

interfaz = widgets.VBox([
    widgets.HTML("<h1>Laboratorio de Crecimiento Poblacional</h1>"),
    controles,
    resultados,
    tablas
])

In [ ]:
def actualizar_interfaz(change=None): 
    # Obtener los valores de los sliders
    valor_r = slider_r.value 
    valor_K = slider_K.value 
    valor_P0 = slider_P0.value 
    valor_t_max = slider_t_max.value 
    valor_h = np.round(slider_h.value, 2)
    mostrar_tablas = checkbox_mostrar_tablas.value

    # Bloquear los sliders mientras se actualiza la interfaz para evitar cambios durante el cálculo
    slider_r.disabled = True
    slider_K.disabled = True
    slider_P0.disabled = True
    slider_t_max.disabled = True
    slider_h.disabled = True

    # Sustituir los valores de los parámetros r y K en la ecuación diferencial
    f_con_parametros = f.subs({ 
        r: valor_r, 
        K: valor_K 
    }) 
    f_con_parametros_numpy = sp.lambdify((t, P), f_con_parametros, 'numpy')

    # Dar formato a la ecuación diferencial con los parámetros sustituidos para mostrarla en la interfaz
    f_str = sp.latex(f)
    f_str = f_str.replace('r ', f'{valor_r}').replace('K', f'{valor_K}').replace('P{\\left(t \\right)}', 'P_i')

    # Limpiar las salidas de los resultados anteriores
    salida_fr_euler.clear_output() 
    salida_fr_RK4.clear_output() 
    salida_grafica.clear_output() 
    salida_euler.clear_output()
    salida_RK4.clear_output()

    # Mostrar la función de recurrencia de Euler con los parámetros sustituidos
    with salida_fr_euler:
        display(Math('P_{i+1}=P_i + ' + f'{valor_h}' + ' \\left(' + f_str + '\\right)'))

    # Mostrar las funciones de recurrencia de RK4 con los parámetros sustituidos
    with salida_fr_RK4:
        display(Math('k1_i=' + f_str.replace('P_i', 'P_i')))
        display(Math('k2_i=' + f_str.replace('P_i', '\\left(P_i + \\frac{'+f'{valor_h}'+'k1_i}{2}\\right)')))
        display(Math('k3_i=' + f_str.replace('P_i', '\\left(P_i + \\frac{'+f'{valor_h}'+'k2_i}{2}\\right)')))
        display(Math('k4_i=' + f_str.replace('P_i', '\\left(P_i + '+f'{valor_h}'+'\\cdot k3_i\\right)')))
        display(Math('P_{i+1}=P_i + \\frac{'+f'{valor_h}'+'}{6}\\left(k1_i + 2k2_i + 2k3_i + k4_i\\right)'))

    # Aplicar método de Euler
    P_euler = euler(0, valor_t_max, valor_P0, valor_h, f_con_parametros_numpy)
    
    # Aplicar método de RK4
    P_RK4 = RK4(0, valor_t_max, valor_P0, valor_h, f_con_parametros_numpy)

    # Aplicar solución analítica
    P_analitica = solucion_analitica(0, valor_t_max, valor_h, valor_P0, valor_K, valor_r)

    # Calcular el error absoluto y porcentual para ambos métodos
    e_a_Euler = error_absoluto(P_euler, P_analitica)
    e_p_Euler = error_porcentual(P_euler, P_analitica)
    e_a_RK4 = error_absoluto(P_RK4, P_analitica)
    e_p_RK4 = error_porcentual(P_RK4, P_analitica)

    # Construir los DataFrames de resultados para ambos metodos
    datos_euler = pd.merge(P_euler, P_analitica[['t', 'P_analitica']], on='t', how='inner')
    datos_euler = pd.merge(datos_euler, e_a_Euler[['t', 'error_absoluto']], on='t', how='inner')
    #datos_euler.rename(columns={'P_numerica': 'P_euler'}, inplace=True)
    datos_euler = pd.merge(datos_euler, e_p_Euler[['t', 'error_porcentual']], on='t', how='inner')
    datos_euler.index.name = 'i'

    datos_RK4 = pd.merge(P_RK4, P_analitica[['t', 'P_analitica']], on='t', how='inner')
    datos_RK4 = pd.merge(datos_RK4, e_a_RK4[['t', 'error_absoluto']], on='t', how='inner')
    datos_RK4 = pd.merge(datos_RK4, e_p_RK4[['t', 'error_porcentual']], on='t', how='inner')
    #datos_RK4.rename(columns={'P_numerica': 'P_RK4'}, inplace=True)
    datos_RK4.index.name = 'i'
    
    # Graficar los resultados y mostrar la gráfica
    fig = graficar(datos_euler, datos_RK4, P_analitica, 0, valor_t_max, valor_h, valor_P0, valor_K, valor_r)
    with salida_grafica:
        display(fig)

    # Mostrar las tablas de resultados para ambos métodos
    if mostrar_tablas:
        with salida_euler:
            display(datos_euler)
        with salida_RK4:
            display(datos_RK4)

    # Desbloquear los sliders después de actualizar la interfaz
    slider_r.disabled = False
    slider_K.disabled = False
    slider_P0.disabled = False
    slider_t_max.disabled = False
    slider_h.disabled = False

In [ ]:
# Configurar los sliders para que llamen a la función actualizar_interfaz cuando su valor cambie
slider_r.observe(actualizar_interfaz, names='value')
slider_K.observe(actualizar_interfaz, names='value')
slider_P0.observe(actualizar_interfaz, names='value')
slider_t_max.observe(actualizar_interfaz, names='value')
slider_h.observe(actualizar_interfaz, names='value')
checkbox_mostrar_tablas.observe(actualizar_interfaz, names='value')

### Laboratorio

Ejecuta todas las celdas anteriores para poder utilizar la interfaz interactiva. Despues ejecuta la siguiente celda para mostrar la interfaz interactiva. Para cambiar los parámetros no es necesario volver a ejecutar la celda, solo se deben mover los sliders y cambiar los valores de los parámetros.

In [ ]:
# Mostrar la interfaz interactiva y actualizarla por primera vez para mostrar los resultados iniciales
display(interfaz)
actualizar_interfaz()

### Ejercicios

1. Configura el laboratorio con los siguientes valores: $r = 0.5$, $K = 2000$, $P(0) = 10$, $\text{Rango de tiempo graficado} = 40$ y $\text{h} = 5$. Se quiere saber cual sera la poblacion en $t=30$ utilizando el método de Euler; modifica el tamaño de paso $h$ hasta obtener un error porcentual aproximado de 10%. ¿Cual es el valor máximo de $h$ que se puede utilizar?
<details>
<summary>Respuesta</summary>

Desde $h=4.5$ el error porcentual de la aproximacion anterior (27) y posterior (31.5) es menor a 10% por lo que se podia hacer una interpolacion entre los 2 valores para calcular el valor en $t=30$. Pero si se quiere un valor de $h$ que llegue exactamente a $t=30$ el valor maximo que cumple es $h=3$. Esto es una de las desventajas de los metodos numericos, para llegar a un valor exacto de $t$ el tamaño de paso debe ser un divisor exacto.

</details>

2. Configura el laboratorio con los siguientes valores: $r = 0.6$, $K = 2000$, $P(0) = 10$ y $\text{Rango de tiempo graficado} = 25$. Varia el tamaño de paso $h$ en el rango $[0, 4]$ observando unicamente el error absoluto de ambos métodos ¿Que similitud encuentras? ¿Como se explica esta similitud?
<details>
<summary>Respuesta</summary>

Mientras menor sea el tamaño de paso aparece un patrón entre los 2 errores, la figura de los errores es similar, ambas tienen la forma de una campana con el pico cercano a $t=10$; la diferencia es la escala de los errores. Este paron aparece porque es la zona donde la población crece más rápido, y es donde los métodos numéricos tienen más dificultad para aproximar la solución analítica. 

Aunque esa es la zona de mayor cambio la escala del error es significativa, con $h=0.1$ el maximo error absoluto del metodo de Euler se da en $t=8.9$ y es $E_a=58.3687$; mientras que el maximo error absoluto del metodo RK4 se da en $t\in [8.4, 8.9]$ y es $E_a=0.0054$. Esto muestra que el metodo RK4 es mucho mas preciso que el metodo de Euler, pero ambos tienen un patron similar.

</details>

3. Configura el laboratorio con los siguientes valores: $r = 0.6$, $K = 2000$, $P(0) = 10$, $\text{Rango de tiempo graficado} = 120$ y $h=3$. Observa la aproximación del método de Euler despues de $t=20$, cuando $t \to 120$ el método de Euler se tiende a $P=2000$. al disminuir el tamaño de paso ¿Cambia el valor al que tiende la aproximación?,ahora aumenta el tamaño de paso ¿Cambia el valor al que tiende la aproximación? ¿Que conclusiones puedes sacar de esto?
<details>
<summary>Respuesta</summary>

Cuando se disminuye el tamaño de paso el valor al que tiende la aproximación no cambia y el error absoluto tiende a 0. Cuando se aumenta el tamaño de paso el valor la aproximación ya no converge a un valor. Cuando se tiene $h=4$ el error relativo ronda entre $E_r \in [20, 40]$. Cuando se usa $h=5$ ya no existe un patrón en la aproximacion y el error ya no se limita a un rango. 

Con esto se puede concluir que existe un tamaño de paso máximo para el método de Euler, si se excede este tamaño de paso la aproximación deja de converger. 

</details>

4. Manteniendo los valores de $r$, $K$, $P(0)$ y el rango de tiempo graficado, varia el valor de $r$ y busca el valor de $h$ donde la aproximacion de Euler deja de converger. ¿Que relación existe entre el valor de $r$ y el tamaño de paso máximo para que la aproximación de Euler converja?
<details>
<summary>Respuesta</summary>

Mientras mayor sea el valor de $r$ el tamaño de paso en el que la aproximación de Euler deja de converger es menor. Mientras menor sea el valor de $r$ el tamaño de paso en el que la aproximación de Euler deja de converger es mayor $r\propto h$.

Para este modelo de crecimiento poblacional logístico el punto de equilibrio es $h\leq \frac{2}{r}$

</details>

## Referencias

Zill, D. G. ., Cullen, M. R. ., García Hernández, A. Elizabeth., & Filio López, Ernesto. (2009). Ecuaciones diferenciales : con problemas de valores en la frontera (7a ed.). Cengage Learning.